In [2]:
from pathlib import Path
import polars as pl
from loguru import logger

BASE_DIR = Path("..")
BRONZE_DIR = BASE_DIR / "data" / "1_bronze"
SILVER_DIR = BASE_DIR / "data" / "2_silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

df_0910 = pl.read_parquet(BRONZE_DIR / "sheet_2009_2010_raw.parquet")
df_1011 = pl.read_parquet(BRONZE_DIR / "sheet_2010_2011_raw.parquet")

logger.info("Nạp dữ liệu thô từ Bronze sang Silver thành công!")

2026-11-05 11:54:53.534 | INFO     | __main__:<module>:13 - Nạp dữ liệu thô từ Bronze sang Silver thành công!


In [65]:
display(df_0910.head(210))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
str,str,str,i64,datetime[ms],f64,str,str
"""489434""","""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,"""13085""","""United Kingdom"""
"""489434""","""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""79323W""",""" WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""22041""","""RECORD FRAME 7"" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,"""13085""","""United Kingdom"""
"""489434""","""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,"""13085""","""United Kingdom"""
…,…,…,…,…,…,…,…
"""C489459""","""90003B""","""ROSE COLOUR PAIR HEART HAIR SL…",-3,2009-12-01 10:44:00,3.75,"""17592""","""United Kingdom"""
"""C489459""","""90209C""","""PINK ENAMEL+GLASS HAIR COMB""",-3,2009-12-01 10:44:00,2.1,"""17592""","""United Kingdom"""
"""C489459""","""90209A""","""PURPLE ENAMEL+GLASS HAIR COMB""",-3,2009-12-01 10:44:00,2.1,"""17592""","""United Kingdom"""


Schema Mapping

In [66]:
SCHEMA_MAPPING = {
    "Customer ID": "CustomerID",
    "Customer_ID": "CustomerID",
    "Price": "UnitPrice",
    "Unit Price": "UnitPrice",
}

df_0910 = df_0910.rename({k: v for k, v in SCHEMA_MAPPING.items() if k in df_0910.columns})
df_1011 = df_1011.rename({k: v for k, v in SCHEMA_MAPPING.items() if k in df_1011.columns})

assert df_0910.columns == df_1011.columns, "Cột chưa khớp!"
logger.success(f"Đã chuẩn hóa tên cột: {df_0910.columns}")

2026-11-04 16:38:03.287 | SUCCESS  | __main__:<module>:12 - Đã chuẩn hóa tên cột: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


white spaces

In [67]:
# Hàm xử lý nghiệp vụ ở tầng Silver: Trim khoảng trắng & Ép kiểu String chuẩn
def trim_and_cast_silver(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("Invoice").cast(pl.Utf8).str.strip_chars(),
        pl.col("StockCode").cast(pl.Utf8).str.strip_chars(),
        pl.col("Description").cast(pl.Utf8).str.strip_chars(),
        pl.col("CustomerID").cast(pl.Utf8).str.strip_chars(),
        pl.col("Country").cast(pl.Utf8).str.strip_chars(),
        pl.col("Quantity").cast(pl.Int64),
        pl.col("UnitPrice").cast(pl.Float64),
        pl.col("InvoiceDate").cast(pl.Datetime)
    ])

df_0910 = trim_and_cast_silver(df_0910)
df_1011 = trim_and_cast_silver(df_1011)

display(df_0910.head(10))

Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,datetime[μs],f64,str,str
"""489434""","""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,"""13085""","""United Kingdom"""
"""489434""","""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""79323W""","""WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,"""13085""","""United Kingdom"""
"""489434""","""22041""","""RECORD FRAME 7"" SINGLE SIZE""",48,2009-12-01 07:45:00,2.1,"""13085""","""United Kingdom"""
"""489434""","""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,"""13085""","""United Kingdom"""
"""489434""","""22064""","""PINK DOUGHNUT TRINKET POT""",24,2009-12-01 07:45:00,1.65,"""13085""","""United Kingdom"""
"""489434""","""21871""","""SAVE THE PLANET MUG""",24,2009-12-01 07:45:00,1.25,"""13085""","""United Kingdom"""
"""489434""","""21523""","""FANCY FONT HOME SWEET HOME DOO…",10,2009-12-01 07:45:00,5.95,"""13085""","""United Kingdom"""
"""489435""","""22350""","""CAT BOWL""",12,2009-12-01 07:46:00,2.55,"""13085""","""United Kingdom"""


Kiểm tra & Đếm chính xác Đơn Hủy 'C'

In [68]:
# Đếm số lượng đơn hủy bắt đầu bằng chữ 'C'
c_0910 = df_0910.filter(pl.col("Invoice").str.starts_with("C")).height
c_1011 = df_1011.filter(pl.col("Invoice").str.starts_with("C")).height

print(f"Đơn hủy (Invoice 'C') - Sheet 2009-2010: {c_0910:,} dòng")
print(f"Đơn hủy (Invoice 'C') - Sheet 2010-2011: {c_1011:,} dòng")

# Hiển thị thử 5 đơn hủy mẫu có chứa chữ C (như mã C489459 trong ảnh của bạn)
print("\n Mẫu 5 đơn hủy thực tế:")
display(df_0910.filter(pl.col("Invoice").str.starts_with("C")).select(["Invoice", "StockCode", "Quantity", "UnitPrice"]).head(5))

Đơn hủy (Invoice 'C') - Sheet 2009-2010: 10,206 dòng
Đơn hủy (Invoice 'C') - Sheet 2010-2011: 9,288 dòng

 Mẫu 5 đơn hủy thực tế:


Invoice,StockCode,Quantity,UnitPrice
str,str,i64,f64
"""C489449""","""22087""",-12,2.95
"""C489449""","""85206A""",-6,1.65
"""C489449""","""21895""",-4,4.25
"""C489449""","""21896""",-6,2.1
"""C489449""","""22083""",-12,2.95


In [69]:
# Lọc loại bỏ đơn hủy
df_0910 = df_0910.filter(~pl.col("Invoice").str.starts_with("C"))
df_1011 = df_1011.filter(~pl.col("Invoice").str.starts_with("C"))

logger.success("Đã lọc loại bỏ toàn bộ các đơn hàng bị hủy ('C').")

2026-11-04 16:38:22.840 | SUCCESS  | __main__:<module>:5 - Đã lọc loại bỏ toàn bộ các đơn hàng bị hủy ('C').


Khảo sát & Làm sạch Số lượng Bán rác (Quantity 
≤ 0)

In [70]:
# Đếm số dòng có Quantity <= 0 (bao gồm hàng mẫu, điều chỉnh kho, quà tặng)
qty_0910 = df_0910.filter(pl.col("Quantity") <= 0).height
qty_1011 = df_1011.filter(pl.col("Quantity") <= 0).height

print(f"Số dòng Quantity <= 0 - Sheet 2009-2010: {qty_0910:,} dòng")
print(f"Số dòng Quantity <= 0 - Sheet 2010-2011: {qty_1011:,} dòng")

# Lọc giữ lại giao dịch bán có Quantity > 0
df_0910 = df_0910.filter(pl.col("Quantity") > 0)
df_1011 = df_1011.filter(pl.col("Quantity") > 0)

Số dòng Quantity <= 0 - Sheet 2009-2010: 2,121 dòng
Số dòng Quantity <= 0 - Sheet 2010-2011: 1,336 dòng


 Khảo sát & Làm sạch Đơn giá Âm / Bằng 0 (UnitPrice $\le$ 0)

In [71]:
# Đếm đơn giá <= 0 (điều chỉnh sổ sách, quà tặng 0đ, lỗi nhập liệu)
price_0910 = df_0910.filter(pl.col("UnitPrice") <= 0).height
price_1011 = df_1011.filter(pl.col("UnitPrice") <= 0).height

print(f" Số dòng UnitPrice <= 0 - Sheet 2009-2010: {price_0910:,} dòng")
print(f" Số dòng UnitPrice <= 0 - Sheet 2010-2011: {price_1011:,} dòng")

# Lọc giữ lại giao dịch có UnitPrice > 0
df_0910 = df_0910.filter(pl.col("UnitPrice") > 0)
df_1011 = df_1011.filter(pl.col("UnitPrice") > 0)

logger.success(" Đã lọc giữ lại toàn bộ giao dịch có UnitPrice > 0.")

2026-11-04 16:39:26.362 | SUCCESS  | __main__:<module>:12 -  Đã lọc giữ lại toàn bộ giao dịch có UnitPrice > 0.


 Số dòng UnitPrice <= 0 - Sheet 2009-2010: 1,569 dòng
 Số dòng UnitPrice <= 0 - Sheet 2010-2011: 1,181 dòng


Xử lý giá trị NULL ở CustomerID & Description (Không thất thoát Demand)

In [ ]:
# Điền CustomerID NULL thành 'GUEST' để giữ nguyên tổng lượng bán sản phẩm
# Điền Description NULL thành 'UNKNOWN DESCRIPTION'
df_0910 = df_0910.with_columns([
    pl.col("CustomerID").fill_null("GUEST"),
    pl.col("Description").fill_null("UNKNOWN DESCRIPTION")
])

df_1011 = df_1011.with_columns([
    pl.col("CustomerID").fill_null("GUEST"),
    pl.col("Description").fill_null("UNKNOWN DESCRIPTION")
])

Kiểm tra lần cuối & Lưu file Parquet Tầng Silver

In [76]:
# Lưu 2 file Parquet sạch độc lập ở tầng Silver
path_0910 = SILVER_DIR / "sheet_2009_2010_cleaned.parquet"
path_1011 = SILVER_DIR / "sheet_2010_2011_cleaned.parquet"

df_0910.write_parquet(path_0910)
df_1011.write_parquet(path_1011)

print("\n" + "="*65)
logger.success("TỔNG KẾT KẾT QUẢ TẦNG SILVER:")
print(f"  └─ Sheet 2009-2010 Sạch: {len(df_0910):,} dòng -> {path_0910}")
print(f"  └─ Sheet 2010-2011 Sạch: {len(df_1011):,} dòng -> {path_1011}")
print("="*65)

2026-11-04 16:41:36.318 | SUCCESS  | __main__:<module>:9 - TỔNG KẾT KẾT QUẢ TẦNG SILVER:



  └─ Sheet 2009-2010 Sạch: 511,565 dòng -> ../data/2_silver/sheet_2009_2010_cleaned.parquet
  └─ Sheet 2010-2011 Sạch: 530,105 dòng -> ../data/2_silver/sheet_2010_2011_cleaned.parquet
